In [0]:
# 1. Configuration
catalog = "capstone6_telecom"
silver_schema = "silver"
gold_schema = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")

def read_silver(table_name):
    return spark.table(f"{catalog}.{silver_schema}.{table_name}")

def save_gold(df, table_name):
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{gold_schema}.{table_name}"))
    print(f"Saved {catalog}.{gold_schema}.{table_name}")

In [0]:
from pyspark.sql import functions as F

In [0]:
# 2. Dimension tables (grain: one row per business key; business attributes only)
# Ingestion lineage and DQ flag columns stay in Silver; dimensions carry only business attributes.
NON_BUSINESS_COLS = ["_rescued_data", "_source_system", "_source_feed", "_source_table", "_source_file",
                     "_ingested_at", "dq_warning_checks"]

def business_cols(df):
    return df.drop(*[c for c in NON_BUSINESS_COLS if c in df.columns])

tower = read_silver("silver_tower_master")
region = read_silver("silver_region_lookup")
cell = read_silver("silver_network_topology_master")

for dim_name, dim_df, dim_key in [("dim_tower", tower, "tower_id"),
                                  ("dim_region", region, "region_id"),
                                  ("dim_cell", cell, "cell_id")]:
    n_rows, n_keys = dim_df.agg(F.count("*"), F.countDistinct(dim_key)).first()
    assert n_rows == n_keys, f"{dim_key} not unique ({n_rows} rows, {n_keys} keys)"
    save_gold(business_cols(dim_df), dim_name)

In [0]:
# 3. Network KPI fact table
kpi = read_silver("silver_network_kpi_measurements")

# Enrich measurements with tower and region attributes where available.
tower_lookup = tower.select("tower_id", "site_name", "region_id", "site_type", "network_generation")

# Join safety: tower_id must be unique in the lookup, otherwise the joins below would multiply records
tower_rows, tower_keys = tower_lookup.agg(F.count("*"), F.countDistinct("tower_id")).first()
assert tower_rows == tower_keys, f"tower_id not unique in silver_tower_master ({tower_rows} rows, {tower_keys} keys)"
# KPI rows already carry their own region_id / network_generation, so tower values are prefixed.
kpi_fact = (kpi
    .join(
        tower_lookup.select(
            "tower_id", "site_name", "site_type",
            F.col("region_id").alias("tower_region_id"),
            F.col("network_generation").alias("tower_network_generation")),
        on="tower_id",
        how="left")
    .withColumn("measurement_date", F.to_date("measurement_ts"))
    # All Silver records are kept, including flagged ones; dq_warning_checks is carried over unchanged.
    .withColumn("is_iqr_outlier", F.coalesce(F.array_contains("dq_warning_checks", "IQR_OUTLIER"), F.lit(False)))
    # KPI region (used for KPI aggregation) compared with the tower master region; mismatches are kept and reported
    .withColumn("kpi_tower_region_match", F.col("region_id") == F.col("tower_region_id")))

save_gold(kpi_fact, "fact_network_kpi")

In [0]:
# 4. Outage fact table
outage = read_silver("silver_network_outage_events")

outage_fact = (outage
    .join(tower_lookup, on="tower_id", how="left")
    .withColumn("outage_date", F.to_date("outage_start_ts"))
    .withColumn(
        "outage_duration_minutes",
        (F.col("outage_end_ts").cast("long") - F.col("outage_start_ts").cast("long")) / 60.0
    ))

save_gold(outage_fact, "fact_network_outage")

In [0]:
# 5. Customer complaint fact table
complaints = read_silver("silver_customer_complaints")

complaint_fact = (complaints
    .join(
        tower.select("tower_id", "site_name", F.col("region_id").alias("tower_region_id")),
        on="tower_id",
        how="left"
    )
    .withColumn("complaint_date", F.to_date("complaint_ts"))
    # Complaints are aggregated by the complaint's own source region (no fallback to the tower region)
    .withColumn("reporting_region_id", F.col("region_id"))
    .drop("tower_region_id"))

save_gold(complaint_fact, "fact_customer_complaint")

In [0]:
# 6. Daily network-quality summary by region
# Region sources: KPI metrics -> KPI's own region_id; outages -> tower master region; complaints -> complaint source region.
# Primary avg_latency_ms excludes measurements flagged IQR_OUTLIER; the unfiltered average is kept for reference.
kpi_daily = (kpi_fact
    .groupBy("region_id", "measurement_date")
    .agg(
        F.count("*").alias("measurement_count"),
        F.avg(F.when(~F.col("is_iqr_outlier"), F.col("latency_ms"))).alias("avg_latency_ms"),
        F.avg("latency_ms").alias("avg_latency_ms_incl_outliers"),
        F.sum(F.col("is_iqr_outlier").cast("int")).alias("outlier_measurement_count"),
        F.avg("packet_loss_pct").alias("avg_packet_loss_pct"),
        F.avg("throughput_mbps").alias("avg_throughput_mbps"),
        F.avg("availability_pct").alias("avg_availability_pct")
    )
    .withColumnRenamed("measurement_date", "summary_date")
    .alias("k"))

outage_daily = (outage_fact
    .groupBy("region_id", "outage_date")
    .agg(
        F.count("*").alias("outage_count"),
        F.avg("outage_duration_minutes").alias("avg_outage_duration_minutes")
    )
    .withColumnRenamed("outage_date", "summary_date")
    .alias("o"))

kpi_outage_daily = (kpi_daily
    .join(
        outage_daily,
        (F.col("k.region_id") == F.col("o.region_id")) &
        (F.col("k.summary_date") == F.col("o.summary_date")),
        "full"
    )
    .select(
        F.coalesce(F.col("k.region_id"), F.col("o.region_id")).alias("region_id"),
        F.coalesce(F.col("k.summary_date"), F.col("o.summary_date")).alias("summary_date"),
        F.col("k.measurement_count"),
        F.col("k.avg_latency_ms"),
        F.col("k.avg_latency_ms_incl_outliers"),
        F.col("k.outlier_measurement_count"),
        F.col("k.avg_packet_loss_pct"),
        F.col("k.avg_throughput_mbps"),
        F.col("k.avg_availability_pct"),
        F.col("o.outage_count"),
        F.col("o.avg_outage_duration_minutes")
    )
    .alias("ko"))

complaint_daily = (complaint_fact
    .groupBy("reporting_region_id", "complaint_date")
    .agg(F.count("*").alias("complaint_count"))
    .withColumnRenamed("reporting_region_id", "region_id")
    .withColumnRenamed("complaint_date", "summary_date")
    .alias("c"))

gold_daily = (kpi_outage_daily
    .join(
        complaint_daily,
        (F.col("ko.region_id") == F.col("c.region_id")) &
        (F.col("ko.summary_date") == F.col("c.summary_date")),
        "full"
    )
    .select(
        F.coalesce(F.col("ko.region_id"), F.col("c.region_id")).alias("region_id"),
        F.coalesce(F.col("ko.summary_date"), F.col("c.summary_date")).alias("summary_date"),
        F.col("ko.measurement_count"),
        F.col("ko.avg_latency_ms"),
        F.col("ko.avg_latency_ms_incl_outliers"),
        F.col("ko.outlier_measurement_count"),
        F.col("ko.avg_packet_loss_pct"),
        F.col("ko.avg_throughput_mbps"),
        F.col("ko.avg_availability_pct"),
        F.col("ko.outage_count"),
        F.col("ko.avg_outage_duration_minutes"),
        F.col("c.complaint_count")
    ))

# Region attributes for reporting (region_id is unique in the region dimension, so rows are not multiplied)
region_attrs_daily = region.select("region_id", "region_name", "service_zone", "sla_tier")
gold_daily = gold_daily.join(F.broadcast(region_attrs_daily), "region_id", "left")

save_gold(gold_daily, "gold_network_quality_daily")

In [0]:
# 7. Outage-to-telemetry correlation and network impact (grain: one row per outage_id)
# KPI measurements and complaints are linked to an outage by tower_id and event time.
# Outages last 5 minutes and measurements are sparse, so an impact window of IMPACT_WINDOW_MINUTES
# before the outage start and after the outage end is used (adjustable assumption).
# Baseline = the same tower's measurements outside all of its outage impact windows.
IMPACT_WINDOW_MINUTES = 60
window_interval = F.expr(f"INTERVAL {IMPACT_WINDOW_MINUTES} MINUTES")

outage_win = (outage_fact
    .select("outage_id", "tower_id", F.col("region_id").alias("tower_region_id"), "site_name", "outage_date",
            "outage_start_ts", "outage_end_ts", "outage_duration_minutes", "outage_type", "severity",
            "root_cause_code", "affected_cells", "incident_status")
    .withColumn("impact_window_start_ts", F.col("outage_start_ts") - window_interval)
    .withColumn("impact_window_end_ts", F.col("outage_end_ts") + window_interval))

kpi_m = kpi_fact.select("measurement_id", F.col("tower_id").alias("m_tower_id"), "measurement_ts", "latency_ms",
                        "packet_loss_pct", "throughput_mbps", "active_sessions", "availability_pct", "is_iqr_outlier")

# One row per (outage, measurement in its impact window)
kpi_pairs = (outage_win.select("outage_id", "tower_id", "outage_start_ts", "outage_end_ts",
                               "impact_window_start_ts", "impact_window_end_ts")
    .join(kpi_m, (F.col("m_tower_id") == F.col("tower_id")) &
                 F.col("measurement_ts").between(F.col("impact_window_start_ts"), F.col("impact_window_end_ts")))
    .withColumn("during_outage", F.col("measurement_ts").between(F.col("outage_start_ts"), F.col("outage_end_ts"))))

def kpi_metrics(prefix):
    # Latency excludes IQR_OUTLIER measurements, consistent with the daily summary
    return [F.avg(F.when(~F.col("is_iqr_outlier"), F.col("latency_ms"))).alias(f"{prefix}_avg_latency_ms"),
            F.avg("packet_loss_pct").alias(f"{prefix}_avg_packet_loss_pct"),
            F.avg("throughput_mbps").alias(f"{prefix}_avg_throughput_mbps"),
            F.avg("active_sessions").alias(f"{prefix}_avg_active_sessions"),
            F.avg("availability_pct").alias(f"{prefix}_avg_availability_pct")]

window_kpi = kpi_pairs.groupBy("outage_id").agg(
    F.count("*").alias("window_measurement_count"),
    F.sum(F.col("during_outage").cast("int")).alias("during_outage_measurement_count"),
    F.sum(F.col("is_iqr_outlier").cast("int")).alias("window_outlier_count"),
    *kpi_metrics("window"))

baseline_kpi = (kpi_m.join(kpi_pairs.select("measurement_id").distinct(), "measurement_id", "left_anti")
    .groupBy(F.col("m_tower_id").alias("tower_id"))
    .agg(F.count("*").alias("baseline_measurement_count"), *kpi_metrics("baseline")))

# Customer impact: complaints on the same tower inside the impact window
cmp_m = complaint_fact.select("complaint_id", F.col("tower_id").alias("c_tower_id"), "complaint_ts", "customer_id")
window_cmp = (outage_win.select("outage_id", "tower_id", "impact_window_start_ts", "impact_window_end_ts")
    .join(cmp_m, (F.col("c_tower_id") == F.col("tower_id")) &
                 F.col("complaint_ts").between(F.col("impact_window_start_ts"), F.col("impact_window_end_ts")))
    .groupBy("outage_id")
    .agg(F.count("*").alias("window_complaint_count"),
         F.countDistinct("customer_id").alias("window_complaining_customers")))

cells_per_tower = cell.groupBy("tower_id").agg(F.count("*").alias("tower_cell_count"))
region_attrs_out = region.select(F.col("region_id").alias("tower_region_id"), "region_name", "service_zone", "sla_tier")

outage_impact = (outage_win
    .join(window_kpi, "outage_id", "left")
    .join(F.broadcast(baseline_kpi), "tower_id", "left")
    .join(window_cmp, "outage_id", "left")
    .join(F.broadcast(cells_per_tower), "tower_id", "left")
    .join(F.broadcast(region_attrs_out), "tower_region_id", "left")
    .fillna(0, ["window_measurement_count", "during_outage_measurement_count", "window_outlier_count",
                "window_complaint_count", "window_complaining_customers", "tower_cell_count",
                "baseline_measurement_count"])
    .withColumn("impact_window_minutes", F.lit(IMPACT_WINDOW_MINUTES))
    .withColumn("latency_delta_ms", F.col("window_avg_latency_ms") - F.col("baseline_avg_latency_ms"))
    .withColumn("availability_delta_pct", F.col("window_avg_availability_pct") - F.col("baseline_avg_availability_pct"))
    .withColumn("packet_loss_delta_pct", F.col("window_avg_packet_loss_pct") - F.col("baseline_avg_packet_loss_pct"))
    .withColumn("throughput_delta_mbps", F.col("window_avg_throughput_mbps") - F.col("baseline_avg_throughput_mbps"))
    .withColumn("affected_cell_share", F.when(F.col("tower_cell_count") > 0,
                                              F.col("affected_cells") / F.col("tower_cell_count")))
    .withColumn("has_telemetry_evidence", F.col("window_measurement_count") > 0)
    .withColumn("has_customer_impact", F.col("window_complaint_count") > 0))

assert outage_impact.count() == outage_fact.count(), "fact_outage_impact must have one row per outage"
save_gold(outage_impact, "fact_outage_impact")

In [0]:
# 8. Network SLA summary (grain: one row per region_id over the reporting period)
# Neither the handbook nor the source data defines targets per sla_tier, so measured service levels are
# reported per region and tier without a pass/fail. Region sources match the daily summary:
# KPI -> KPI region_id, outages -> tower master region, complaints -> complaint source region.
valid_latency = F.when(~F.col("is_iqr_outlier"), F.col("latency_ms"))

sla_kpi = kpi_fact.groupBy("region_id").agg(
    F.count("*").alias("measurement_count"),
    F.countDistinct("measurement_date").alias("measurement_days"),
    F.avg("availability_pct").alias("avg_availability_pct"),
    F.min("availability_pct").alias("min_availability_pct"),
    F.avg(valid_latency).alias("avg_latency_ms"),
    F.percentile_approx(valid_latency, 0.95, 10000).alias("p95_latency_ms"),
    F.avg("packet_loss_pct").alias("avg_packet_loss_pct"),
    F.avg("throughput_mbps").alias("avg_throughput_mbps"),
    F.sum(F.col("is_iqr_outlier").cast("int")).alias("outlier_measurement_count"))

sla_outage = outage_impact.groupBy(F.col("tower_region_id").alias("region_id")).agg(
    F.count("*").alias("outage_count"),
    F.sum("outage_duration_minutes").alias("total_outage_minutes"),
    F.sum("affected_cells").alias("total_affected_cells"),
    F.sum(F.col("has_customer_impact").cast("int")).alias("outages_with_complaints"),
    F.sum(F.col("has_telemetry_evidence").cast("int")).alias("outages_with_telemetry"))

sla_complaint = complaint_fact.groupBy(F.col("reporting_region_id").alias("region_id")).agg(
    F.count("*").alias("complaint_count"))

sla_summary = (business_cols(region)
    .select("region_id", "region_name", "state", "country", "service_zone", "market_segment", "sla_tier", "active_flag")
    .join(sla_kpi, "region_id", "left")
    .join(sla_outage, "region_id", "left")
    .join(sla_complaint, "region_id", "left")
    .fillna(0, ["measurement_count", "measurement_days", "outlier_measurement_count", "outage_count",
                "total_outage_minutes", "total_affected_cells", "outages_with_complaints",
                "outages_with_telemetry", "complaint_count"]))

assert sla_summary.count() == region.count(), "gold_region_sla_summary must have one row per region"
save_gold(sla_summary, "gold_region_sla_summary")

# Service levels per SLA tier (measurement-weighted averages)
display(kpi_fact.join(F.broadcast(region.select("region_id", "sla_tier")), "region_id", "left")
        .groupBy("sla_tier")
        .agg(F.countDistinct("region_id").alias("regions_with_kpi"),
             F.count("*").alias("measurement_count"),
             F.round(F.avg("availability_pct"), 3).alias("avg_availability_pct"),
             F.round(F.avg(valid_latency), 2).alias("avg_latency_ms"),
             F.round(F.percentile_approx(valid_latency, 0.95, 10000), 2).alias("p95_latency_ms"),
             F.round(F.avg("packet_loss_pct"), 3).alias("avg_packet_loss_pct"))
        .join(sla_summary.groupBy("sla_tier").agg(F.sum("outage_count").alias("outage_count"),
                                                   F.sum("complaint_count").alias("complaint_count")),
              "sla_tier", "full")
        .orderBy("sla_tier"))

In [0]:
# 8. Updated daily network-quality summary
daily = spark.table(f"{catalog}.{gold_schema}.gold_network_quality_daily")
display(daily.orderBy("summary_date", "region_id"))

# Region-days whose primary latency average changed because outliers were excluded
display(daily.filter(F.col("outlier_measurement_count") > 0)
             .select("region_id", "summary_date", "measurement_count", "outlier_measurement_count",
                     "avg_latency_ms", "avg_latency_ms_incl_outliers")
             .orderBy("summary_date", "region_id"))